# Customer Churn Prediction Pipeline – End-to-End Documentation

In [0]:
from pyspark.sql import SparkSession
import random

spark = SparkSession.builder.getOrCreate()

# Sample values
contract_types = ["Monthly", "Yearly", "Quarterly"]
churn_values = ["Yes", "No"]

data = []

for i in range(1, 501):  # 500 records
    customer_id = f"C{i:03d}"
    age = random.randint(18, 65)
    tenure = random.randint(1, 60)
    monthly_charges = random.randint(200, 1000)
    contract_type = random.choice(contract_types)
    support_calls = random.randint(0, 10)
    
    # Simple churn logic (for realistic pattern)
    churn = "Yes" if (support_calls > 5 and contract_type == "Monthly") else random.choice(churn_values)
    
    data.append((customer_id, age, tenure, monthly_charges, contract_type, support_calls, churn))

# Create DataFrame
columns = ["customer_id", "age", "tenure", "monthly_charges", "contract_type", "support_calls", "churn"]

df = spark.createDataFrame(data, columns)

# Save as Delta Table
df.write.format("delta").mode("overwrite").saveAsTable("churn_dataset")

# Show sample
df.show(10)

In [0]:
%sql
CREATE CATALOG usecase_three

In [0]:
%sql
USE CATALOG usecase_three;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS customer_data

In [0]:
%sql
USE CATALOG usecase_three;
USE SCHEMA customer_data;

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("usecase_three.customer_data.churn_dataset")

In [0]:
from pyspark.sql.functions import col

silver_df = (
    spark.table("usecase_three.customer_data.churn_dataset")
    .dropDuplicates()
    .withColumn("monthly_charges", col("monthly_charges").cast("int"))
)

silver_df.write.format("delta").mode("overwrite").saveAsTable("usecase_three.customer_data.silver_churn")

In [0]:
from pyspark.sql.functions import when

gold_df = (
    spark.table("usecase_three.customer_data.silver_churn")
    .withColumn("is_high_support", when(col("support_calls") > 3, 1).otherwise(0))
    .withColumn("churn_label", when(col("churn") == "Yes", 1).otherwise(0))
)

gold_df.write.format("delta").mode("overwrite").saveAsTable("usecase_three.customer_data.gold_churn")

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import mlflow

pdf = spark.table("usecase_three.customer_data.gold_churn").toPandas()

X = pdf[["age", "tenure", "monthly_charges", "support_calls"]]
y = pdf["churn_label"]

X_train, X_test, y_train, y_test = train_test_split(X, y)

with mlflow.start_run():
    model = RandomForestClassifier()
    model.fit(X_train, y_train)
    accuracy = model.score(X_test, y_test)
    
    mlflow.log_metric("accuracy", accuracy)

In [0]:
df = spark.table("churn_dataset")

df.count()              # Should be 200
df.printSchema()        # Check data types
df.show(10)             # Sample data

In [0]:
spark.table("gold_churn").groupBy("contract_type", "churn").count().show()